In [ ]:
# imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
# AMP components (GradScaler, autocast) are removed
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
)
from PIL import Image
import os
import random
import copy
import time
import logging # Logging
from tqdm import tqdm # Use standard tqdm for console

In [ ]:
# --- Configuration ---
# Path to your FER2013 CSV file
CSV_PATH = "../../fer2013.csv"
# Model/Checkpoint saving directory
local_time = time.localtime()
current_date = time.strftime("%Y-%m-%d", local_time)
MODEL_DIR = "models_checkpointed"
# Log file path
LOG_FILE = "training_log.log"
os.makedirs(MODEL_DIR, exist_ok=True)

# --- Basic Logging Setup ---
# Remove existing handlers if any to avoid duplicate logging on re-runs in some environments
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode='w'), # mode='w' to overwrite log file on each run
        logging.StreamHandler()
    ]
)
logging.info(f"Logging to console and {LOG_FILE}")

# --- Reproducibility ---
def set_seed(seed=42):
    """Sets the seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    logging.info(f"Random seed set to {seed}")

SEED = 42
set_seed(SEED)

# --- Training Hyperparameters ---
BASE_MODEL_NAME = "VGG11"
BATCH_SIZE = 16
EPOCHS_PHASE1 = 7 # Increased
EPOCHS_PHASE2 = 20 # Increased
K_FOLDS = 5
LEARNING_RATE_HEAD = 1e-3
LEARNING_RATE_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-2
SCHEDULER_PATIENCE = 3
SCHEDULER_FACTOR = 0.1
EARLY_STOPPING_PATIENCE = 7 # Increased
DROPOUT_RATE = 0.5

# --- Log Hyperparameters ---
logging.info("--- Hyperparameters ---")
logging.info(f"Base Model: {BASE_MODEL_NAME}")
logging.info(f"Batch Size: {BATCH_SIZE}")
logging.info(f"Epochs Phase 1 (Head Training): {EPOCHS_PHASE1}")
logging.info(f"Epochs Phase 2 (Full Model Fine-tuning): {EPOCHS_PHASE2}")
logging.info(f"K-Folds for Cross-Validation: {K_FOLDS}")
logging.info(f"Learning Rate (Head): {LEARNING_RATE_HEAD}")
logging.info(f"Learning Rate (Backbone): {LEARNING_RATE_BACKBONE}")
logging.info(f"Weight Decay: {WEIGHT_DECAY}")
logging.info(f"Scheduler Patience: {SCHEDULER_PATIENCE}")
logging.info(f"Scheduler Factor: {SCHEDULER_FACTOR}")
logging.info(f"Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
logging.info(f"Dropout Rate: {DROPOUT_RATE}")
logging.info(f"Random Seed: {SEED}")
logging.info(f"AMP (Mixed Precision): False") # Explicitly state AMP is off
logging.info("-----------------------")

In [ ]:
# --- Load Data ---
logging.info(f"Loading data from {CSV_PATH}")
try:
    df = pd.read_csv(CSV_PATH)
except FileNotFoundError:
    logging.error(f"Error: CSV file not found at {CSV_PATH}")
    exit()
except Exception as e:
    logging.error(f"Error loading CSV file: {e}")
    exit()

df_test = df[df["Usage"] == "PrivateTest"].copy()
df_train_val = df[df["Usage"].isin(["Training", "PublicTest"])].copy()
logging.info(f"Loaded {len(df_train_val)} samples for Training/Validation")
logging.info(f"Loaded {len(df_test)} samples for Final Testing")

df_train_val["label"] = (df_train_val["emotion"] == 3).astype(int)
df_test["label"] = (df_test["emotion"] == 3).astype(int)

class_counts = df_train_val["label"].value_counts().sort_index()
if len(class_counts) < 2 or 0 in class_counts and class_counts.get(0, 0) == 0 or 1 in class_counts and class_counts.get(1, 0) == 0:
    logging.warning("One or both classes have zero samples in training/validation data or only one class present. Using equal class weights [1.0, 1.0].")
    class_weights_tensor = torch.tensor([1.0, 1.0], dtype=torch.float32)
else:
    total_samples = len(df_train_val)
    class_weights = total_samples / (len(class_counts) * class_counts)
    class_weights_tensor = torch.tensor(class_weights.values, dtype=torch.float32)
logging.info(f"Class counts (Train/Val): {class_counts.to_dict()}")
logging.info(f"Calculated class weights: {class_weights_tensor.numpy()}")


In [ ]:

# --- Dataset Class ---
class FERDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            pixels = np.fromiter(row["pixels"].split(" "), dtype=np.uint8).reshape(48, 48)
        except Exception as e:
            logging.error(f"Error processing row {idx} with label {row.get('label', 'N/A')}: {row['pixels'][:50]}... Error: {e}")
            raise e
        img = Image.fromarray(pixels).convert("RGB")
        label = row["label"]
        if self.transform:
            img = self.transform(img)
        return img, label

# --- Transforms ---
train_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

val_test_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


In [ ]:

# --- Model Definition ---
def get_model(dropout_rate_param=DROPOUT_RATE):
    logging.info(f"Initializing {BASE_MODEL_NAME} model.")
    model = models.vgg11(weights=models.VGG11_Weights.DEFAULT)
    
    if hasattr(model, 'features') and hasattr(model, 'classifier'):
        # VGG13's classifier structure:
        # (classifier): Sequential(
        #   (0): Linear(in_features=25088, out_features=4096, bias=True)
        #   (1): ReLU(inplace=True)
        #   (2): Dropout(p=0.5, inplace=False)
        #   (3): Linear(in_features=4096, out_features=4096, bias=True)
        #   (4): ReLU(inplace=True)
        #   (5): Dropout(p=0.5, inplace=False)
        #   (6): Linear(in_features=4096, out_features=1000, bias=True)
        # )
        # Adjust the existing dropout layers' rates
        dropout_indices = [i for i, layer in enumerate(model.classifier) if isinstance(layer, nn.Dropout)]
        if len(dropout_indices) > 0:
            for idx in dropout_indices:
                model.classifier[idx].p = dropout_rate_param
            logging.info(f"Adjusted VGG11's existing dropout rates at classifier indices {dropout_indices} to {dropout_rate_param}")
        else:
            logging.warning("Could not find Dropout layers in VGG13.classifier to adjust rate. Structure might have changed.")

        # Replace the final linear layer
        # Find the last linear layer to replace
        last_linear_idx = -1
        for i, layer in reversed(list(enumerate(model.classifier))):
            if isinstance(layer, nn.Linear):
                last_linear_idx = i
                break
        
        if last_linear_idx != -1:
            num_ftrs_final_layer = model.classifier[last_linear_idx].in_features
            model.classifier[last_linear_idx] = nn.Linear(num_ftrs_final_layer, 2) # Binary classification
            logging.info(f"Replaced final layer of {BASE_MODEL_NAME} (classifier[{last_linear_idx}]) for 2 output classes.")
        else:
            logging.error(f"Could not find the final Linear layer in {BASE_MODEL_NAME}'s classifier to replace.")
            # Fallback or raise error
            num_ftrs_final_layer = 4096 # Common VGG output before final FC
            model.classifier[-1] = nn.Linear(num_ftrs_final_layer, 2)
            logging.warning(f"Attempting to replace classifier[-1] as a fallback.")

    else:
        logging.error(f"Model {BASE_MODEL_NAME} does not have expected 'features' or 'classifier' attributes.")
        # Handle error appropriately, e.g., raise an exception
        raise AttributeError(f"Model {BASE_MODEL_NAME} structure not as expected.")
    return model

In [ ]:
# --- Training & Evaluation Functions ---
def train_one_epoch(
    model, loader, criterion, optimizer, device, phase="Head" # scaler removed
):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader, desc=f"Training Epoch ({phase})", leave=False)
    for i, (imgs, labels) in enumerate(pbar):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)

        try:
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        except Exception as e:
            logging.error(f"Error during forward/loss calculation in training batch {i}: {e}")
            continue

        try:
            loss.backward()
            optimizer.step()
        except Exception as e:
             logging.error(f"Error during backward/step in training batch {i}: {e}")
             optimizer.zero_grad(set_to_none=True)
             continue

        if not torch.isnan(loss) and not torch.isinf(loss):
            running_loss += loss.item() * imgs.size(0)
        else:
            logging.warning(f"NaN or Inf loss detected in training batch {i}. Skipping accumulation.")
        pbar.set_postfix(loss=loss.item() if not torch.isnan(loss) else float('nan'))

    if not loader.dataset or len(loader.dataset) == 0: return 0.0
    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss

def evaluate(model, loader, criterion, device): # scaler removed
    model.eval()
    all_preds, all_targets = [], []
    all_probs = []
    running_loss = 0.0
    pbar = tqdm(loader, desc="Evaluating", leave=False)
    with torch.no_grad():
        for i, (imgs, labels) in enumerate(pbar):
            imgs, labels = imgs.to(device), labels.to(device)
            try:
                outputs = model(imgs)
                loss = criterion(outputs, labels)

                if not torch.isnan(loss) and not torch.isinf(loss):
                    running_loss += loss.item() * imgs.size(0)
                else:
                    logging.warning(f"NaN or Inf loss detected in evaluation batch {i}. Skipping accumulation.")

                _, predicted = torch.max(outputs, 1)
                probabilities = torch.softmax(outputs, dim=1)[:, 1]

                all_preds.extend(predicted.cpu().numpy())
                all_targets.extend(labels.cpu().numpy())
                all_probs.extend(probabilities.cpu().numpy())
            except Exception as e:
                logging.error(f"Error during evaluation batch {i}: {e}")
                continue

    if not loader.dataset or not all_targets:
        logging.warning("Evaluation dataset empty or all batches failed.")
        return {
            "loss": float('inf'), "accuracy": 0.0, "f1": 0.0, "precision": 0.0,
            "recall": 0.0, "auc": 0.0, "cm": np.zeros((2, 2)), "report": {},
            "preds": np.array([]), "targets": np.array([])
        }

    val_loss = running_loss / len(all_targets) if len(all_targets) > 0 else float('inf')
    all_targets_np = np.array(all_targets)
    all_preds_np = np.array(all_preds)
    all_probs_np = np.array(all_probs)

    val_acc = accuracy_score(all_targets_np, all_preds_np) if len(all_targets_np) > 0 else 0.0
    val_f1 = f1_score(all_targets_np, all_preds_np, average="binary", zero_division=0) if len(all_targets_np) > 0 else 0.0
    val_prec = precision_score(all_targets_np, all_preds_np, average="binary", zero_division=0) if len(all_targets_np) > 0 else 0.0
    val_rec = recall_score(all_targets_np, all_preds_np, average="binary", zero_division=0) if len(all_targets_np) > 0 else 0.0
    val_auc = 0.0
    try:
        if len(np.unique(all_targets_np)) > 1: # Check for at least two classes for AUC
             val_auc = roc_auc_score(all_targets_np, all_probs_np)
        elif len(all_targets_np) > 0: # Only one class present
             logging.warning("AUC calculation skipped: only one class present in evaluation targets.")
    except ValueError as e: # Catch specific sklearn error if all predictions are one class
        logging.warning(f"AUC calculation failed (ValueError): {e}")
    except Exception as e: # Catch any other unexpected error
        logging.error(f"AUC calculation failed (General Error): {e}")


    cm = confusion_matrix(all_targets_np, all_preds_np) if len(all_targets_np) > 0 else np.zeros((2,2))
    report_dict = {}
    if len(all_targets_np) > 0:
        try:
            report_dict = classification_report(
                    all_targets_np, all_preds_np, target_names=["Not Happy", "Happy"], output_dict=True, zero_division=0
                )
        except Exception as e:
            logging.error(f"Error generating classification report: {e}")

    metrics = {
        "loss": val_loss, "accuracy": val_acc, "f1": val_f1, "precision": val_prec,
        "recall": val_rec, "auc": val_auc, "cm": cm, "report": report_dict,
        "preds": all_preds_np, "targets": all_targets_np
    }
    return metrics

# --- Main K-Fold Cross-Validation ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"Using device: {device}")
num_workers = 4 if os.name != "nt" and torch.cuda.is_available() else 0
logging.info(f"Using {num_workers} workers for DataLoaders")

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
X_data = df_train_val.index.values
y_data = df_train_val["label"].values

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_data)):
    logging.info(f"--- Starting Fold {fold+1}/{K_FOLDS} ---")
    fold_start_time = time.time()

    train_df = df_train_val.iloc[train_idx]
    val_df = df_train_val.iloc[val_idx]

    train_ds = FERDataset(train_df, transform=train_transform)
    val_ds = FERDataset(val_df, transform=val_test_transform)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True
    )

    model = get_model().to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
    # scaler is removed as AMP is not used

    # --- Phase 1: Train the Head ---
    logging.info("--- Phase 1: Training Head ---")
    if hasattr(model, 'features'): # VGG has 'features' for convolutional layers
        for param in model.features.parameters():
            param.requires_grad = False
    else:
        logging.warning(f"{BASE_MODEL_NAME} does not have 'features' attribute. Cannot freeze backbone by this name.")
    # Ensure classifier parameters require grad
    if hasattr(model, 'classifier'):
        for param in model.classifier.parameters():
            param.requires_grad = True
    else:
        logging.warning(f"{BASE_MODEL_NAME} does not have 'classifier' attribute.")


    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), # Only pass parameters that require grad
        lr=LEARNING_RATE_HEAD,
        weight_decay=WEIGHT_DECAY
    )

    for epoch in range(EPOCHS_PHASE1):
        epoch_start_time = time.time()
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device, phase="Head" # scaler removed
        )
        epoch_time = time.time() - epoch_start_time
        logging.info(
            f"Fold {fold+1} Phase 1 - Epoch {epoch+1}/{EPOCHS_PHASE1}, Train Loss: {train_loss:.4f}, Time: {epoch_time:.2f}s"
        )

    # --- Phase 2: Fine-tune the whole model ---
    logging.info("--- Phase 2: Fine-tuning Full Model ---")
    if hasattr(model, 'features'):
        for param in model.features.parameters():
            param.requires_grad = True
    # Classifier params should already be requires_grad=True

    optimizer = optim.AdamW(
        [
            {
                "params": model.features.parameters() if hasattr(model, 'features') else [], # Handle if no 'features'
                "lr": LEARNING_RATE_BACKBONE,
            },
            {
                "params": model.classifier.parameters() if hasattr(model, 'classifier') else model.parameters(), # Fallback if no 'classifier'
                "lr": LEARNING_RATE_HEAD,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = ReduceLROnPlateau(
        optimizer, mode='min', factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE, verbose=False
    )

    best_val_f1 = -1.0
    patience_counter = 0
    best_epoch = -1
    checkpoint_path = os.path.join(MODEL_DIR, f"best_model_fold_{fold+1}.pth")

    for epoch in range(EPOCHS_PHASE2):
        epoch_start_time = time.time()
        current_epoch_total = EPOCHS_PHASE1 + epoch + 1

        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device, phase="Full" # scaler removed
        )
        val_metrics = evaluate(model, val_loader, criterion, device) # scaler removed

        epoch_time = time.time() - epoch_start_time
        current_lrs = [group['lr'] for group in optimizer.param_groups]
        logging.info(
            f"Fold {fold+1} Phase 2 - Epoch {epoch+1}/{EPOCHS_PHASE2} (Total: {current_epoch_total}), "
            f"LR: {current_lrs}, Train Loss: {train_loss:.4f}, Val Loss: {val_metrics['loss']:.4f}, "
            f"Val Acc: {val_metrics['accuracy']:.4f}, Val F1: {val_metrics['f1']:.4f}, "
            f"Val AUC: {val_metrics['auc']:.4f}, Time: {epoch_time:.2f}s"
        )

        scheduler.step(val_metrics['loss'])

        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            patience_counter = 0
            best_epoch = current_epoch_total
            try:
                torch.save({
                    'epoch': current_epoch_total,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'loss': val_metrics['loss'],
                    'f1': best_val_f1,
                }, checkpoint_path)
                logging.info(
                    f"  -> New best F1: {best_val_f1:.4f} at epoch {best_epoch}. Checkpoint saved to {checkpoint_path}"
                )
            except Exception as e:
                 logging.error(f"Error saving checkpoint: {e}")
        else:
            patience_counter += 1
            logging.info(f"  -> F1 did not improve. Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            logging.info(f"  -> Early stopping triggered at epoch {current_epoch_total}.")
            break

    if os.path.exists(checkpoint_path):
        logging.info(f"Loading best model from {checkpoint_path} (Epoch {best_epoch}, F1: {best_val_f1:.4f})")
        try:
            checkpoint = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
        except Exception as e:
            logging.error(f"Error loading checkpoint: {e}. Using last model state.")
    else:
        logging.warning("No best model checkpoint found for this fold. Using last model state.")

    logging.info(f"--- Evaluating Best Model for Fold {fold+1} ---")
    final_fold_metrics = evaluate(model, val_loader, criterion, device) # scaler removed

    logging.info(f"Fold {fold+1} Final Validation Results (Best Model):")
    logging.info(f"  Accuracy:  {final_fold_metrics['accuracy']:.4f}")
    logging.info(f"  F1 Score:  {final_fold_metrics['f1']:.4f}")
    logging.info(f"  Precision: {final_fold_metrics['precision']:.4f}")
    logging.info(f"  Recall:    {final_fold_metrics['recall']:.4f}")
    logging.info(f"  AUC:       {final_fold_metrics['auc']:.4f}")
    logging.info(f"  Loss:      {final_fold_metrics['loss']:.4f}")
    logging.info("  Confusion Matrix:")
    logging.info(f"\n{final_fold_metrics['cm']}")

    fold_results.append(final_fold_metrics)
    fold_time = time.time() - fold_start_time
    logging.info(f"--- Fold {fold+1} completed in {fold_time:.2f}s ---")

In [ ]:
# --- Cross-Validation Summary ---
logging.info("--- Cross-Validation Summary ---")
if fold_results:
    accs = [r["accuracy"] for r in fold_results]
    f1s = [r["f1"] for r in fold_results]
    precs = [r["precision"] for r in fold_results]
    recs = [r["recall"] for r in fold_results]
    aucs = [r["auc"] for r in fold_results]

    logging.info(f"Mean Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    logging.info(f"Mean F1 Score:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    logging.info(f"Mean Precision: {np.mean(precs):.4f} ± {np.std(precs):.4f}")
    logging.info(f"Mean Recall:    {np.mean(recs):.4f} ± {np.std(recs):.4f}")
    logging.info(f"Mean AUC:       {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
else:
    logging.warning("No fold results to summarize.")

# --- Training Final Model on Full Train/Val Data ---
logging.info("--- Training Final Model on Full Train/Val Dataset ---")
final_model = get_model().to(device)
final_ds = FERDataset(df_train_val, transform=train_transform)
final_loader = DataLoader(
    final_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True
)
criterion_final = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
# scaler_final is removed

# --- Final Model - Phase 1: Train the Head ---
logging.info("--- Final Model - Phase 1: Training Head ---")
if hasattr(final_model, 'features'):
    for param in final_model.features.parameters():
        param.requires_grad = False
if hasattr(final_model, 'classifier'):
    for param in final_model.classifier.parameters():
        param.requires_grad = True

optimizer_head_final = optim.AdamW(
    filter(lambda p: p.requires_grad, final_model.parameters()),
    lr=LEARNING_RATE_HEAD,
    weight_decay=WEIGHT_DECAY
)
for epoch in range(EPOCHS_PHASE1):
    epoch_start_time = time.time()
    train_loss = train_one_epoch(
        final_model, final_loader, criterion_final, optimizer_head_final, device, phase="Head" # scaler removed
    )
    epoch_time = time.time() - epoch_start_time
    logging.info(
        f"Final Phase 1 - Epoch {epoch+1}/{EPOCHS_PHASE1}, Train Loss: {train_loss:.4f}, Time: {epoch_time:.2f}s"
    )

# --- Final Model - Phase 2: Fine-tune the whole model ---
logging.info("--- Final Model - Phase 2: Fine-tuning Full Model ---")
if hasattr(final_model, 'features'):
    for param in final_model.features.parameters():
        param.requires_grad = True
# Classifier params should already be requires_grad=True

optimizer_full_final = optim.AdamW(
    [
        {
            "params": final_model.features.parameters() if hasattr(final_model, 'features') else [],
            "lr": LEARNING_RATE_BACKBONE,
        },
        {
            "params": final_model.classifier.parameters() if hasattr(final_model, 'classifier') else final_model.parameters(),
            "lr": LEARNING_RATE_HEAD,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)
for epoch in range(EPOCHS_PHASE2):
    epoch_start_time = time.time()
    train_loss = train_one_epoch(
        final_model, final_loader, criterion_final, optimizer_full_final, device, phase="Full" # scaler removed
    )
    epoch_time = time.time() - epoch_start_time
    logging.info(
        f"Final Phase 2 - Epoch {epoch+1}/{EPOCHS_PHASE2}, Train Loss: {train_loss:.4f}, Time: {epoch_time:.2f}s"
    )

final_model_save_path = os.path.join(MODEL_DIR, f"{BASE_MODEL_NAME.lower()}_fer2013_happy_final.pth")
try:
    torch.save(final_model.state_dict(), final_model_save_path)
    logging.info(f"Final model state_dict saved to {final_model_save_path}")
except Exception as e:
    logging.error(f"Error saving final model: {e}")

In [ ]:
# --- Final Evaluation on Test Set ---
logging.info("--- Evaluating Final Model on PrivateTest Set ---")
if len(df_test) > 0:
    test_ds = FERDataset(df_test, transform=val_test_transform)
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True
    )

    try:
        eval_model = get_model().to(device) # Create a new model instance for evaluation
        eval_model.load_state_dict(torch.load(final_model_save_path, map_location=device))
        eval_model.eval() # Ensure model is in eval mode
        logging.info(f"Successfully loaded final model from {final_model_save_path}")

        # 1. Standard Metrics Evaluation
        logging.info("--- Calculating Performance Metrics on Test Set ---")
        test_metrics = evaluate(eval_model, test_loader, criterion_final, device)

        logging.info("Final Test Set Performance Metrics:")
        logging.info(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
        logging.info(f"  F1 Score:  {test_metrics['f1']:.4f}")
        logging.info(f"  Precision: {test_metrics['precision']:.4f}")
        logging.info(f"  Recall:    {test_metrics['recall']:.4f}")
        logging.info(f"  AUC:       {test_metrics['auc']:.4f}")
        logging.info(f"  Loss:      {test_metrics['loss']:.4f}")
        logging.info("  Confusion Matrix:")
        logging.info(f"\n{test_metrics['cm']}")
        logging.info("  Classification Report:")
        if test_metrics['targets'].size > 0:
             logging.info(f"\n{classification_report(test_metrics['targets'], test_metrics['preds'], target_names=['Not Happy', 'Happy'], zero_division=0)}")
        else:
             logging.warning("No predictions generated during final test evaluation for report.")

        # 2. Inference Speed Test
        logging.info("--- Performing Inference Speed Test on Test Set ---")
        dummy_batch = None
        try:
            dummy_batch = next(iter(test_loader)) # Get one batch for warm-up
        except StopIteration:
            logging.warning("Test loader is empty, cannot perform warm-up for speed test.")
        
        if dummy_batch:
            dummy_imgs, _ = dummy_batch
            dummy_imgs = dummy_imgs.to(device)

            # Warm-up iterations
            logging.info("Performing warm-up inferences...")
            with torch.no_grad():
                for _ in range(5): # Number of warm-up iterations
                    _ = eval_model(dummy_imgs)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
            logging.info("Warm-up complete.")

        total_inference_time = 0
        total_images_processed = 0
        inference_times_per_batch = []

        logging.info("Starting timed inferences...")
        with torch.no_grad():
            for imgs, _ in tqdm(test_loader, desc="Inference Speed Test"):
                imgs = imgs.to(device)
                
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                start_time = time.perf_counter() # More precise timer

                _ = eval_model(imgs)

                if device.type == 'cuda':
                    torch.cuda.synchronize()
                end_time = time.perf_counter()

                batch_time = end_time - start_time
                inference_times_per_batch.append(batch_time)
                total_inference_time += batch_time
                total_images_processed += imgs.size(0)
        
        if total_images_processed > 0:
            avg_time_per_image = total_inference_time / total_images_processed
            images_per_second = total_images_processed / total_inference_time
            avg_time_per_batch = total_inference_time / len(test_loader)

            logging.info("Inference Speed Test Results:")
            logging.info(f"  Total images processed: {total_images_processed}")
            logging.info(f"  Total inference time: {total_inference_time:.4f} seconds")
            logging.info(f"  Average inference time per batch: {avg_time_per_batch:.6f} seconds (Batch Size: {BATCH_SIZE})")
            logging.info(f"  Average inference time per image: {avg_time_per_image:.6f} seconds")
            logging.info(f"  Images Per Second (IPS): {images_per_second:.2f}")
        else:
            logging.warning("No images were processed during the inference speed test (Test set might be empty or all batches failed).")

    except FileNotFoundError:
        logging.error(f"Final model file not found at {final_model_save_path}. Cannot evaluate on test set.")
    except Exception as e:
        logging.error(f"Error during final evaluation or speed test on test set: {e}", exc_info=True)

else:
    logging.warning("PrivateTest set is empty or could not be loaded. Skipping final evaluation.")

In [ ]:
# --- Optimization Notes ---
logging.info("--- Optimization Notes ---")
logging.info("The saved final model is a standard PyTorch state dictionary.")
logging.info("For near real-time performance as discussed in the methodology,")
logging.info("further optimization steps like conversion to ONNX, TensorRT, or OpenVINO,")
logging.info("and potentially quantization, would be necessary after training.")

logging.info("--- Script Finished ---")